# บทที่ 8 — แล้ว RUL ล่ะ — จาก Health Index สู่ 'เหลืออีกกี่ชั่วโมง'

<sub>บทเรียนที่ 8 จาก 8 &nbsp;·&nbsp; [← บทที่ 7](07_reading_results_critically.ipynb) · [สารบัญ](README.md)</sub>

## เป้าหมายของบทนี้

เมื่อจบบทนี้คุณจะ:

- เข้าใจความต่างระหว่าง Health Index กับ RUL
- เห็นด้วยตัวเองว่าการแบ่งข้อมูลผิดวิธีทำให้ตัวเลขสวยได้ขนาดไหน
- แปลง Health Index ที่ทำนายได้ให้เป็นเวลาที่เหลือ
- รู้จักเมตริกของวงการ PHM ที่ลงโทษการเตือนช้ามากกว่าเตือนเร็ว

> **บทนี้ไม่ต้องใช้ข้อมูลดิบ** — ใช้ไฟล์ `outputs/features_cache.npz` ที่อยู่ใน repo อยู่แล้ว รันได้เลย

---

## 8.1 สิ่งที่เรายังไม่ได้ทำ

ตลอด 7 บทที่ผ่านมา เราทำนาย **Health Index** — ตัวเลข 0 ถึง 1 ที่บอกว่าลูกปืนสุขภาพดีแค่ไหน

แต่ชื่อโปรเจกต์คือ *RUL Prediction* และ **RUL (Remaining Useful Life)** คือ
**"เหลืออีกกี่ชั่วโมงก่อนพัง"** ซึ่งไม่ใช่สิ่งเดียวกัน

| | Health Index | RUL |
|---|---|---|
| หน่วย | ไม่มีหน่วย (0–1) | ชั่วโมง / รอบการทำงาน |
| ตอบคำถาม | "ตอนนี้สภาพเป็นอย่างไร" | "เหลือเวลาอีกเท่าไร" |
| ใช้ตัดสินใจ | ต้องแปลงก่อน | วางแผนซ่อมได้ทันที |

วิศวกรโรงงานไม่ได้อยากรู้ว่า HI = 0.38 เขาอยากรู้ว่า
**"ยังใช้ได้อีก 3 วัน หรือต้องหยุดเครื่องพรุ่งนี้"**

บทนี้จะปิดช่องว่างนั้น — และระหว่างทางจะเจอบทเรียนที่สำคัญที่สุดของทั้งซีรีส์

In [ ]:
# ── ตั้งค่าให้ notebook มองเห็นโค้ดใน src/ ──
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# ── หาฟอนต์ที่แสดงภาษาไทยได้ ไม่งั้นข้อความในกราฟจะกลายเป็นสี่เหลี่ยม ──
_installed = {f.name for f in fm.fontManager.ttflist}
for _f in ["Noto Sans Thai", "Leelawadee UI", "Tahoma", "TH Sarabun New", "Angsana New"]:
    if _f in _installed:
        plt.rcParams["font.family"] = _f
        plt.rcParams["axes.unicode_minus"] = False    # ฟอนต์ไทยมักไม่มีเครื่องหมายลบแบบ unicode
        print("ฟอนต์กราฟ:", _f)
        break
else:
    print("[หมายเหตุ] ไม่พบฟอนต์ไทย - ข้อความไทยในกราฟอาจแสดงเป็นสี่เหลี่ยม")
    print("           Windows/macOS มักมีอยู่แล้ว ส่วน Linux ลง: sudo apt install fonts-thai-tlwg")

print("project root:", ROOT)

In [ ]:
import torch
import torch.nn as nn
import copy

from src.paths import FEATURES_CACHE
from src.data_loader import compute_health_index
from src.dataset import split_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"

features = np.load(FEATURES_CACHE)["features"]
N = len(features)
STEP_MINUTES = 10       # บันทึกทุก 10 นาที

print("จำนวนจุดเวลา:", N)
print(f"ระยะเวลาทดลอง: {N * STEP_MINUTES / 60:.1f} ชั่วโมง ({N * STEP_MINUTES / 60 / 24:.1f} วัน)")

## 8.2 ข้อมูลชุดนี้มี RUL จริงอยู่แล้ว

ข่าวดี: เพราะการทดลองนี้เดินจน**ลูกปืนพังจริง** เราจึงรู้ว่าจุดจบอยู่ตรงไหน

```
RUL ณ จุดเวลา t = (จุดสุดท้าย − t) × 10 นาที
```

นี่คือ RUL จริง ไม่ใช่การสมมติ

In [ ]:
timesteps = np.arange(N)
rul_hours = (N - 1 - timesteps) * STEP_MINUTES / 60.0
rul_norm = (rul_hours / rul_hours.max()).astype(np.float32)   # ปรับเป็น 0-1

hi = compute_health_index(features, window=7)

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

axes[0].plot(rul_hours, color="#d32f2f", linewidth=2)
axes[0].set_ylabel("RUL (ชั่วโมง)")
axes[0].set_title("RUL จริง - ลดลงเป็นเส้นตรงตามนิยาม เพราะเวลาเดินสม่ำเสมอ")

axes[1].plot(hi, color="#2e7d32", linewidth=1.6)
axes[1].set_ylabel("Health Index")
axes[1].set_xlabel("จุดเวลา")
axes[1].set_title("Health Index - นิ่งนาน แล้วทรุดเร็วช่วงท้าย")

plt.tight_layout()
plt.show()

print(f"RUL ที่จุดเริ่มต้น: {rul_hours[0]:.1f} ชั่วโมง")
print(f"HI ต่ำสุดที่จุดเวลา {hi.argmin()} (เหลืออีก {rul_hours[hi.argmin()]:.1f} ชั่วโมง)")

**ดูสองกราฟนี้ให้ดี — นี่คือรากของปัญหาทั้งหมด**

- กราฟบน (RUL) ลดลงเป็นเส้นตรงสม่ำเสมอ เพราะเวลาเดินไปข้างหน้าเท่ากันเสมอ
- กราฟล่าง (HI) นิ่งอยู่นานมาก แล้วค่อยทรุดเฉพาะช่วงท้าย

โมเดลมองเห็นแต่สัญญาณสั่น ซึ่งสอดคล้องกับกราฟล่าง แต่เราอยากให้มันทำนายกราฟบน

## 8.3 การทดลองที่จะเปลี่ยนวิธีอ่านผลของคุณไปตลอด

มาทดลองจริง 4 แบบ — 2 label × 2 วิธีแบ่งข้อมูล:

| | สุ่มสลับ (shuffle) | แบ่งตามเวลา |
|---|---|---|
| **Health Index** | ? | ? |
| **RUL** | ? | ? |

ลองเดาก่อนรันว่าช่องไหนจะได้คะแนนดีที่สุด แล้วค่อยดูคำตอบ

In [ ]:
class SimpleLSTM(nn.Module):
    def __init__(self, n_features=56):
        super().__init__()
        self.lstm1 = nn.LSTM(n_features, 128, batch_first=True)
        self.lstm2 = nn.LSTM(128, 64, batch_first=True)
        self.fc1 = nn.Linear(64, 32)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x):
        x, _ = self.lstm1(x)
        x, _ = self.lstm2(x)
        x = x[:, -1, :]
        return torch.sigmoid(self.fc2(torch.relu(self.fc1(x)))).squeeze(-1)


def r2_score(t, p):
    return 1 - np.sum((t - p) ** 2) / np.sum((t - t.mean()) ** 2)


def run(labels, shuffle, epochs=40, patience=10, seed=42):
    tr, va, te, _ = split_dataset(features, labels, window_size=20, batch_size=32,
                                  shuffle_split=shuffle, random_seed=seed)
    torch.manual_seed(seed)
    model = SimpleLSTM().to(device)
    loss_fn = nn.MSELoss()
    opt = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)

    best, best_state, wait = float("inf"), None, 0
    for _ in range(epochs):
        model.train()
        for xb, yb in tr:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        model.eval()
        tot = 0.0
        with torch.no_grad():
            for xb, yb in va:
                xb, yb = xb.to(device), yb.to(device)
                tot += loss_fn(model(xb), yb).item() * len(xb)
        v = tot / len(va.dataset)
        if v < best:
            best, best_state, wait = v, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_state)
    model.eval()
    P, T = [], []
    with torch.no_grad():
        for xb, yb in te:
            P.append(model(xb.to(device)).cpu().numpy())
            T.append(yb.numpy())
    P, T = np.concatenate(P), np.concatenate(T)
    return {"r2": r2_score(T, P), "rmse": float(np.sqrt(np.mean((P - T) ** 2))),
            "preds": P, "targets": T}


print("เริ่มทดลอง (ใช้เวลาราว 2 นาที)\n")
out = {}
for lname, lab in [("Health Index", hi), ("RUL", rul_norm)]:
    for sname, sh in [("สุ่มสลับ", True), ("แบ่งตามเวลา", False)]:
        out[(lname, sname)] = run(lab, sh)
        print(f"  {lname:<14} {sname:<14} -> R2 = {out[(lname, sname)]['r2']:>10.4f}")

In [ ]:
print(f"\n{'':<16}{'สุ่มสลับ':>16}{'แบ่งตามเวลา':>18}")
print("=" * 52)
for lname in ["Health Index", "RUL"]:
    a = out[(lname, "สุ่มสลับ")]["r2"]
    b = out[(lname, "แบ่งตามเวลา")]["r2"]
    print(f"{lname:<16}{a:>16.4f}{b:>18.4f}")
print()
print("R2 = 1.0 คือสมบูรณ์แบบ | R2 = 0 คือพอ ๆ กับทายค่าเฉลี่ย | ติดลบ = แย่กว่าทายค่าเฉลี่ย")

## 8.4 อ่านตารางนี้ให้ดี

ผลที่ได้มักจะออกมาแบบนี้:

- **RUL + สุ่มสลับ** ได้ R² สูงมาก เกือบ 1.0 — ดูเหมือนโมเดลเก่งสุด ๆ
- **RUL + แบ่งตามเวลา** ได้ R² **ติดลบมหาศาล** — แย่กว่าทายค่าเฉลี่ยหลายสิบเท่า

**โมเดลตัวเดียวกัน ข้อมูลชุดเดียวกัน ต่างกันแค่วิธีแบ่ง แต่ผลต่างกันคนละโลก**

### ทำไมถึงเป็นแบบนั้น

ย้อนกลับไปบทที่ 4 — หน้าต่างที่อยู่ติดกันใช้ข้อมูลร่วมกัน 19 จาก 20 จุด

พอสุ่มสลับ หน้าต่างที่ 100 อยู่ใน train ส่วนหน้าต่างที่ 101 อยู่ใน test
โมเดลเห็นแล้วว่าหน้าต่าง 100 มี RUL = 147.2 ชั่วโมง
พอเจอหน้าต่าง 101 ที่หน้าตาเกือบเหมือนกันเป๊ะ มันก็แค่**ตอบค่าใกล้ ๆ กัน**

**นี่ไม่ใช่การทำนาย แต่คือการเปิดดูเฉลยของเพื่อนบ้าน**

RUL ยิ่งโดนหนักกว่า HI เพราะ RUL เป็นฟังก์ชันของเวลาล้วน ๆ
การรู้ RUL ของเพื่อนบ้านจึงเท่ากับรู้คำตอบเลย

In [ ]:
# พิสูจน์: ในโหมดสุ่มสลับ หน้าต่างใน test มีเพื่อนบ้านอยู่ใน train กี่ %
n_win = N - 20 + 1
rng = np.random.default_rng(42)
idx_all = np.arange(n_win)
rng.shuffle(idx_all)
n_tr, n_va = int(n_win * 0.70), int(n_win * 0.15)
tr_set = set(idx_all[:n_tr].tolist())
te_idx = idx_all[n_tr + n_va:]

has_neighbour = sum(1 for i in te_idx if (i - 1) in tr_set or (i + 1) in tr_set)
print(f"หน้าต่างใน test ที่มีเพื่อนบ้านติดกันอยู่ใน train: "
      f"{has_neighbour} / {len(te_idx)} ({has_neighbour/len(te_idx)*100:.0f}%)")
print()
print("หน้าต่างที่ติดกันซ้อนทับกัน 19/20 จุด = เกือบเป็นข้อมูลเดียวกัน")
print("-> โมเดลแค่ interpolate จากคำตอบที่เห็นแล้วก็ได้คะแนนเกือบเต็ม")

In [ ]:
# ดูให้เห็นภาพ: ทำนาย RUL ในสองโหมด
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

for ax, sname in zip(axes, ["สุ่มสลับ", "แบ่งตามเวลา"]):
    d = out[("RUL", sname)]
    ax.scatter(d["targets"], d["preds"], s=14, alpha=0.55, color="#d32f2f")
    lims = [0, 1]
    ax.plot(lims, lims, "k--", linewidth=1.2)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel("RUL จริง (ปรับเป็น 0-1)")
    ax.set_ylabel("RUL ที่ทำนาย")
    ax.set_title(f"RUL + {sname}\nR2 = {d['r2']:.3f}")

plt.tight_layout()
plt.show()

กราฟซ้ายจุดเกาะเส้นทแยงเป๊ะ กราฟขวากระจัดกระจาย —
**และกราฟขวาคือความจริงว่าโมเดลทำนายอนาคตได้แค่ไหน**

> ### ⚠️ ข้อสรุปที่ต้องจำ
>
> ผลหลักของโปรเจกต์นี้ (บทที่ 6) ใช้การแบ่งแบบสุ่มสลับ
> ตัวเลขที่ได้จึง**ดีเกินจริงไปพอสมควร**
>
> เหตุผลที่ยังใช้ได้คือเป้าหมายของโปรเจกต์คือ**เปรียบเทียบสถาปัตยกรรม**
> ซึ่งทุกโมเดลเจอเงื่อนไขเดียวกันหมด การเทียบจึงยังยุติธรรม
>
> แต่ถ้าจะเคลมว่า "ระบบนี้เอาไปใช้จริงได้" ต้องรายงานตัวเลขจากการแบ่งตามเวลา

In [ ]:
# แล้ว Health Index ล่ะ - โดนผลกระทบเหมือนกันไหม
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

for ax, sname in zip(axes, ["สุ่มสลับ", "แบ่งตามเวลา"]):
    d = out[("Health Index", sname)]
    ax.scatter(d["targets"], d["preds"], s=14, alpha=0.55, color="#2e7d32")
    lo = min(d["targets"].min(), d["preds"].min())
    up = max(d["targets"].max(), d["preds"].max())
    ax.plot([lo, up], [lo, up], "k--", linewidth=1.2)
    ax.set_xlabel("Health Index จริง")
    ax.set_ylabel("Health Index ที่ทำนาย")
    ax.set_title(f"Health Index + {sname}\nR2 = {d['r2']:.3f}")

plt.tight_layout()
plt.show()

print("Health Index ก็โดนเหมือนกัน แต่ไม่รุนแรงเท่า RUL")
print("เพราะ HI ผูกกับสภาพของสัญญาณ ไม่ได้ผูกกับเวลาโดยตรง")

## 8.5 แล้วจะทำนาย RUL อย่างไรให้ถูกวิธี

ถ้าทำนาย RUL ตรง ๆ ไม่ได้ วงการ PHM ใช้วิธีสามขั้น:

```
1. ทำนาย Health Index จากสัญญาณ            ← ทำไปแล้วในบท 1-6
2. กำหนดเกณฑ์ว่า HI เท่าไรถือว่า "พัง"
3. ประมาณว่าอีกนานแค่ไหน HI จะลงถึงเกณฑ์นั้น   ← ได้ RUL
```

ข้อดีคือขั้นที่ 1 ใช้สิ่งที่สัญญาณบอกได้จริง ส่วนขั้นที่ 3 เป็นการคำนวณตรงไปตรงมา
ไม่ได้ขอให้โมเดลเดาเวลาจากข้อมูลที่ไม่มีข้อมูลเรื่องเวลา

In [ ]:
# กำหนดเกณฑ์ "พัง" แบบทนต่อสัญญาณรบกวน
# ใช้จุดที่ HI ลงต่ำกว่าเกณฑ์แล้วไม่กลับขึ้นมาอีก (sustained crossing)
# ถ้าใช้ "ครั้งแรกที่ต่ำกว่าเกณฑ์" เฉย ๆ จะโดนหลอกด้วยการแกว่งช่วงต้น

PCT = 5
FAIL_THRESHOLD = float(np.percentile(hi, PCT))

sustained = [t for t in range(N) if np.all(hi[t:] <= FAIL_THRESHOLD)]
FAIL_TIME = sustained[0] if sustained else int(np.argmin(hi))

print(f"เกณฑ์ว่าพัง (percentile {PCT} ของ HI): {FAIL_THRESHOLD:.4f}")
print(f"จุดที่ HI ลงต่ำกว่าเกณฑ์แล้วไม่กลับขึ้นอีก: t = {FAIL_TIME}")
print(f"  = ชั่วโมงที่ {FAIL_TIME*STEP_MINUTES/60:.1f} จากทั้งหมด {N*STEP_MINUTES/60:.1f}")
print(f"  = เตือนล่วงหน้าได้ {(N-1-FAIL_TIME)*STEP_MINUTES/60:.1f} ชั่วโมงก่อนจบการทดลอง")

In [ ]:
TREND_WINDOW = 30      # ดูแนวโน้มจาก 30 จุดล่าสุด (5 ชั่วโมง)


def estimate_rul(hi_series, t, threshold=FAIL_THRESHOLD, window=TREND_WINDOW):
    """ประมาณ RUL ที่จุดเวลา t โดยต่อเส้นแนวโน้มของ HI ออกไปหาเกณฑ์

    คืนค่าเป็นชั่วโมง (np.inf ถ้าแนวโน้มยังไม่ลดลง = ยังไม่เห็นสัญญาณพัง)
    """
    lo = max(0, t - window + 1)
    seg = hi_series[lo:t + 1]
    if len(seg) < 3:
        return np.inf

    x = np.arange(len(seg))
    slope, intercept = np.polyfit(x, seg, 1)      # fit เส้นตรง

    if slope >= -1e-9:                            # ยังไม่ลดลง
        return np.inf

    current = slope * (len(seg) - 1) + intercept
    steps_left = (current - threshold) / (-slope)
    return max(0.0, steps_left * STEP_MINUTES / 60.0)


est = np.array([estimate_rul(hi, t) for t in range(N)])
true_rul = np.maximum(0.0, (FAIL_TIME - np.arange(N)) * STEP_MINUTES / 60.0)

finite = np.isfinite(est)
print(f"ประมาณค่าได้ {finite.sum()} จาก {N} จุดเวลา")
print("(จุดที่ประมาณไม่ได้คือช่วงที่ HI ยังไม่มีแนวโน้มลดลง - ซึ่งถูกต้องแล้ว)")

In [ ]:
plt.figure(figsize=(11, 4.5))
plt.plot(true_rul, label="RUL จริง (จนถึงจุดที่ถือว่าพัง)", color="#111", linewidth=2.2)
plt.plot(np.where(finite, est, np.nan), label="RUL ที่ประมาณจากแนวโน้ม HI",
         color="#d32f2f", linewidth=1.1, alpha=0.85)
plt.axvline(FAIL_TIME, color="#f57c00", linestyle="--", linewidth=1.5, label="จุดที่ถือว่าพัง")
plt.ylim(0, 60)
plt.xlabel("จุดเวลา")
plt.ylabel("RUL (ชั่วโมง)")
plt.title("ประมาณ RUL จากแนวโน้มของ Health Index")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# วัดความแม่นเฉพาะช่วงที่ใกล้พังจริง - ช่วงที่การทำนายมีความหมาย
HORIZON = 12    # ชั่วโมง
zone = finite & (true_rul <= HORIZON) & (true_rul > 0)

if zone.sum() == 0:
    print("ไม่มีจุดที่ประมาณค่าได้ในช่วงนี้ - ลองเพิ่ม HORIZON")
else:
    err = est[zone] - true_rul[zone]
    print(f"ประเมินในช่วง {HORIZON} ชั่วโมงสุดท้ายก่อนถึงเกณฑ์ ({zone.sum()} จุด)")
    print("=" * 56)
    print(f"  ค่าคลาดเคลื่อนเฉลี่ย (MAE) : {np.mean(np.abs(err)):>6.2f} ชั่วโมง")
    print(f"  ค่าคลาดเคลื่อนกลาง        : {np.median(np.abs(err)):>6.2f} ชั่วโมง")
    print(f"  เตือนเร็วเกินไป            : {int(np.sum(err < 0)):>3} จุด")
    print(f"  เตือนช้าเกินไป             : {int(np.sum(err > 0)):>3} จุด")

## 8.6 ⚠️ ผิดพลาดสองแบบ ราคาไม่เท่ากัน

ในงานบำรุงรักษา ความผิดพลาดสองทิศทางมีต้นทุนต่างกันมาก

| ประเภท | เกิดอะไรขึ้น | ผลที่ตามมา |
|---|---|---|
| **ทำนายต่ำกว่าจริง**<br>(เตือนเร็ว) | บอกว่าเหลือ 5 ชม. จริง ๆ เหลือ 20 | เปลี่ยนชิ้นส่วนเร็วไป เสียค่าอะไหล่ |
| **ทำนายสูงกว่าจริง**<br>(เตือนช้า) | บอกว่าเหลือ 20 ชม. จริง ๆ เหลือ 5 | **เครื่องพังกลางคัน** สายการผลิตหยุด |

แบบล่างแพงกว่ามาก แต่ RMSE ธรรมดาลงโทษทั้งสองแบบเท่ากัน

วงการ PHM จึงใช้ **asymmetric scoring** สูตรที่นิยมมาจาก PHM08 Challenge

In [ ]:
def phm_score(error_hours):
    """คะแนนแบบ PHM08 - ยิ่งน้อยยิ่งดี

    error = ทำนาย - จริง
      error < 0 (เตือนเร็ว) หารด้วย 13  -> ลงโทษเบา
      error > 0 (เตือนช้า)  หารด้วย 10  -> ลงโทษหนักกว่า
    """
    e = np.asarray(error_hours, dtype=float)
    return float(np.sum(np.where(e < 0, np.exp(-e / 13.0) - 1, np.exp(e / 10.0) - 1)))


print("ค่าปรับสำหรับความผิดพลาดขนาดเท่ากัน")
print("=" * 46)
print(f"{'ผิดพลาด':>12}{'ค่าปรับ':>12}   ประเภท")
for e in [-20, -10, -5, 0, 5, 10, 20]:
    tag = "เตือนเร็ว" if e < 0 else ("เตือนช้า" if e > 0 else "ตรงเป๊ะ")
    print(f"{e:>+9} ชม.{phm_score([e]):>11.1f}   {tag}")

In [ ]:
e_range = np.linspace(-30, 30, 300)
penalty = np.where(e_range < 0, np.exp(-e_range / 13.0) - 1, np.exp(e_range / 10.0) - 1)

plt.figure(figsize=(10, 4))
plt.plot(e_range, penalty, color="#d32f2f", linewidth=2.2, label="PHM08 score")
plt.plot(e_range, e_range ** 2 / 40, color="#1976d2", linewidth=1.8,
         linestyle="--", label="MSE (ปรับสเกลให้เทียบได้)")
plt.axvline(0, color="#888", linewidth=1)
plt.fill_betweenx([0, penalty.max()], -30, 0, color="#4caf50", alpha=0.07)
plt.fill_betweenx([0, penalty.max()], 0, 30, color="#f44336", alpha=0.07)
plt.text(-15, penalty.max() * 0.8, "เตือนเร็ว\n(ยอมรับได้)", ha="center", color="#2e7d32")
plt.text(15, penalty.max() * 0.8, "เตือนช้า\n(อันตราย)", ha="center", color="#c62828")
plt.xlabel("ความผิดพลาด (ชั่วโมง)  =  ทำนาย − จริง")
plt.ylabel("ค่าปรับ")
plt.title("PHM08 ลงโทษการเตือนช้าหนักกว่า ต่างจาก MSE ที่สมมาตร")
plt.legend()
plt.ylim(0, penalty.max())
plt.tight_layout()
plt.show()

## 8.7 สรุป — RUL อยู่ตรงไหนในโปรเจกต์นี้

**สิ่งที่โปรเจกต์นี้ทำจริง:** ทำนาย Health Index จากสัญญาณสั่น

**สิ่งที่บทนี้เพิ่ม:** แปลง HI เป็น RUL ด้วยการต่อเส้นแนวโน้ม
พร้อมพิสูจน์ว่าทำไมการทำนาย RUL ตรง ๆ ถึงไม่ควรเชื่อ

**ข้อจำกัดที่ต้องพูดให้ชัด:**

1. **"จุดพัง" ถูกนิยามเอง** — ใช้ percentile ของ HI เป็นเกณฑ์
   งานจริงต้องมาจากมาตรฐานวิศวกรรมหรือข้อมูลความเสียหายจริง

2. **ประมาณได้เฉพาะช่วงท้าย** — ตอน HI ยังนิ่ง การต่อเส้นแนวโน้มไม่มีความหมาย
   ซึ่งสะท้อนความจริงว่าสัญญาณยังไม่มีข้อมูลเรื่องเวลาที่เหลือ

3. **ทดสอบบน run เดียว** — เรารู้จุดจบเพราะการทดลองนี้จบด้วยความเสียหาย
   ถ้ามีหลาย run จะประเมินได้น่าเชื่อถือกว่ามาก

4. **การต่อเส้นตรงเป็นสมมติฐานหยาบ** — การเสื่อมช่วงท้ายมักเป็นเลขชี้กำลัง

> **สิ่งที่ควรเขียนในรายงาน**
>
> ❌ "โมเดลทำนาย RUL ได้ RMSE 0.008"
>
> ✅ "โมเดลทำนาย Health Index ได้ R² = 0.94 ภายใต้การแบ่งข้อมูลแบบสุ่มสลับ
> (ซึ่งมี optimistic bias จากหน้าต่างที่ซ้อนทับกัน) และเมื่อแปลงเป็น RUL
> ด้วยการต่อเส้นแนวโน้ม ให้ค่าคลาดเคลื่อนเฉลี่ย X ชั่วโมงในช่วง 12 ชั่วโมงสุดท้าย"
>
> ยาวกว่า แต่**ตรงกับสิ่งที่ทำจริง** และไม่มีใครมาจับผิดได้ทีหลัง

## 🔧 ลองแก้ดู — ต่อยอดเรื่อง RUL


1. เปลี่ยน `PCT = 5` เป็น 2 และ 10 — จุดที่ถือว่าพังเลื่อนไปเท่าไร?
   RUL ที่ประมาณได้เปลี่ยนตามแค่ไหน? บอกอะไรเรื่องความไวต่อการเลือกเกณฑ์?
2. เปลี่ยน `TREND_WINDOW` เป็น 10 และ 60 — สั้นไวต่อสัญญาณรบกวน ยาวตอบสนองช้า
   จุดสมดุลอยู่ตรงไหน?
3. ลองเปลี่ยน `np.polyfit(x, seg, 1)` เป็น degree 2 หรือ fit บน `np.log(seg)`
4. **ท้าทาย:** ใช้ HI ที่**โมเดลทำนาย** (ไม่ใช่ HI จริง) มาประมาณ RUL —
   นี่คือสิ่งที่เกิดขึ้นจริงตอนใช้งาน ความคลาดเคลื่อนสะสมแค่ไหน?
5. **สำคัญ:** ลองรันบทที่ 6 ใหม่ทั้งหมดโดยเปลี่ยนเป็น `shuffle_split=False`
   แล้วดูว่าอันดับของ 4 สถาปัตยกรรมเปลี่ยนไปไหม

## ❓ เช็คความเข้าใจ

**1. ทำไมการทำนาย RUL แบบสุ่มสลับถึงได้ R² เกือบ 1.0 ทั้งที่จริง ๆ ทำไม่ได้?**

<details>
<summary>ดูเฉลย</summary>

เพราะหน้าต่างที่ติดกันซ้อนทับกัน 19/20 จุด เมื่อสุ่มสลับ หน้าต่างใน test เกือบทุกอัน มีเพื่อนบ้านอยู่ใน train ที่โมเดลรู้คำตอบแล้ว โมเดลจึงแค่ interpolate ไม่ได้ทำนายจริง พอเปลี่ยนเป็นแบ่งตามเวลาที่ไม่มีเพื่อนบ้านให้ลอก ผลก็พังทันที

</details>

**2. ทำไม RUL ถึงโดนผลกระทบจาก leakage หนักกว่า Health Index?**

<details>
<summary>ดูเฉลย</summary>

เพราะ RUL เป็นฟังก์ชันของเวลาล้วน ๆ การรู้ RUL ของหน้าต่างข้างเคียงเท่ากับรู้คำตอบเลย ส่วน HI ผูกกับสภาพของสัญญาณ ซึ่งโมเดลยังต้องเรียนรู้ความสัมพันธ์ระหว่าง feature กับสภาพอยู่บ้าง

</details>

**3. ทำไมค่าปรับของ PHM08 ถึงไม่สมมาตร?**

<details>
<summary>ดูเฉลย</summary>

เพราะการเตือนช้าทำให้เครื่องพังกลางคัน ซึ่งต้นทุนสูงกว่าการเปลี่ยนชิ้นส่วนเร็วไปมาก เมตริกควรสะท้อนต้นทุนจริงในการใช้งาน ไม่ใช่แค่วัดระยะห่างทางคณิตศาสตร์

</details>

**4. ถ้าอยากทำนาย RUL ได้แม่นตั้งแต่ต้นการทดลอง ต้องเพิ่มอะไร?**

<details>
<summary>ดูเฉลย</summary>

ต้องมีข้อมูลที่บอกอายุได้ เช่น จำนวนรอบหมุนสะสม ประวัติภาระงาน อุณหภูมิสะสม หรือมีข้อมูลจากหลาย run เพื่อให้โมเดลเรียนรู้การกระจายของอายุ ลำพังสัญญาณสั่น ณ ขณะนั้นไม่มีข้อมูลว่าผ่านมากี่ชั่วโมงแล้ว

</details>

---

## สรุปบทนี้

- Health Index บอกสภาพ ส่วน RUL บอกเวลาที่เหลือ — ไม่ใช่สิ่งเดียวกัน
- การแบ่งข้อมูลผิดวิธีทำให้ R² กระโดดจากติดลบเป็นเกือบ 1.0 ได้ โดยที่โค้ดไม่มี error
- ทำนาย RUL ตรง ๆ จากสัญญาณไม่เวิร์ค เพราะสัญญาณช่วงปกติไม่มีข้อมูลเรื่องเวลา
- วิธีมาตรฐานคือทำนาย HI ก่อน แล้วต่อเส้นแนวโน้มไปหาเกณฑ์ที่ถือว่าพัง
- ในงานบำรุงรักษา เตือนช้าแพงกว่าเตือนเร็ว เมตริกจึงควรไม่สมมาตร
- รายงานผลให้ตรงกับสิ่งที่ทำจริง พร้อมระบุวิธีแบ่งข้อมูลเสมอ

[← บทที่ 7](07_reading_results_critically.ipynb) &nbsp;·&nbsp; [สารบัญ](README.md)